In [11]:
import os
import pandas as pd
from datetime import datetime

# 실제 환경 기준 경로 설정
target_folder = r"C:\ai_x\source\Pks_Develop\N시기별음식URL수집\N월별조회메뉴수집_비중반영"
save_folder = r"C:\ai_x\source\Pks_Develop\N시기별음식URL수집\N월별조회메뉴수집_비중반영_예측가능메뉴한정"
os.makedirs(save_folder, exist_ok=True)

# 예측 가능한 메뉴 리스트
target_menus = [
    '바비큐립', '바게트', '반미', '팥빙수', '불고기', '분짜', '치즈버거', '부리또', '치즈케이크',
    '후라이드치킨', '쿠키', '크루아상', '크로크무슈', '레드커리', '딤섬', '에그베네딕트',
    '감자튀김', '프렌치토스트', '돼지갈비', '참치김밥', '그라탱', '훠궈', '짜장면', '잡채밥',
    '케밥', '김치찌개', '해물전', '라자냐', '마카롱', '마파두부밥', '머핀', '나초', '팟타이',
    '팬케이크', '마라파스타', '한국식피자', '케사디아', '라멘', '쌀국수', '해물리조또',
    '시저샐러드', '생선회', '미역국', '소바', '크림수프', '스테이크와감자', '연어초밥',
    '타코야키', '국물떡볶이', '우동'
]

# 처리 로그
results = []

for file in os.listdir(target_folder):
    if not file.endswith(".xlsx"):
        continue

    file_path = os.path.join(target_folder, file)
    try:
        xls = pd.ExcelFile(file_path)
        sheet_names = xls.sheet_names

        # 시트 자동 인식
        summary_sheet = next((s for s in sheet_names if '요약' in s), None)
        url_sheet = next((s for s in sheet_names if 'url' in s.lower()), None)

        if not summary_sheet or not url_sheet:
            results.append((file, "시트 누락"))
            continue

        df_url = pd.read_excel(xls, sheet_name=url_sheet)
        df_summary = pd.read_excel(xls, sheet_name=summary_sheet)

        if '메뉴' not in df_url.columns or '메뉴' not in df_summary.columns:
            results.append((file, "'메뉴' 열 없음"))
            continue

        df_url_filtered = df_url[df_url['메뉴'].isin(target_menus)].copy()
        df_summary['처리 내역'] = df_summary['메뉴'].apply(lambda x: '예측대상' if x in target_menus else '제외')
        df_summary_filtered = df_summary[df_summary['메뉴'].isin(target_menus)].copy()

        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        base_name = os.path.splitext(file)[0]
        new_filename = f"{base_name}_{timestamp}.xlsx"
        save_path = os.path.join(save_folder, new_filename)

        with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
            df_url_filtered.to_excel(writer, sheet_name='url목록', index=False)
            df_summary_filtered.to_excel(writer, sheet_name='요약', index=False)

        results.append((file, "저장 완료"))

    except Exception as e:
        results.append((file, f"오류: {str(e)}"))

# 결과 로그 출력
df_log = pd.DataFrame(results, columns=["파일명", "처리 결과"])
print(df_log)

                                      파일명  처리 결과
0   menu_images_202306_20250717_1418.xlsx  저장 완료
1   menu_images_202307_20250717_1958.xlsx  저장 완료
2   menu_images_202308_20250718_0127.xlsx  저장 완료
3   menu_images_202309_20250717_1415.xlsx  저장 완료
4   menu_images_202310_20250717_1955.xlsx  저장 완료
5   menu_images_202311_20250718_0125.xlsx  저장 완료
6   menu_images_202312_20250717_1419.xlsx  저장 완료
7   menu_images_202401_20250717_2003.xlsx  저장 완료
8   menu_images_202402_20250718_0134.xlsx  저장 완료
9   menu_images_202403_20250717_1417.xlsx  저장 완료
10  menu_images_202404_20250717_2000.xlsx  저장 완료
11  menu_images_202405_20250718_0130.xlsx  저장 완료
12  menu_images_202406_20250717_0433.xlsx  저장 완료
13  menu_images_202407_20250717_0920.xlsx  저장 완료
14  menu_images_202408_20250717_1510.xlsx  저장 완료
15  menu_images_202409_20250717_2055.xlsx  저장 완료
16  menu_images_202410_20250718_0219.xlsx  저장 완료
17  menu_images_202411_20250717_1352.xlsx  저장 완료
18  menu_images_202412_20250717_1921.xlsx  저장 완료
19  menu_images_2025